# Báo cáo Phân tích Dữ liệu OULAD & Đánh giá So sánh Mô hình Huấn luyện

**Dự án**: Intelligent Student Advisor Platform (Capstone)  
**Mô hình được đánh giá**: `oulad-risk-logistic-regression` (Phiên bản: `final-002`)  
**Bộ dữ liệu**: Open University Learning Analytics Dataset (OULAD)  
**Mục tiêu**: Dự đoán nguy cơ sinh viên Không hoàn thành môn học (`non_completion_at_end`: Fail hoặc Withdrawn) tại mốc **Ngày 28** (kết thúc 4 tuần đầu).  
**Mục đích sổ tay**: Cung cấp số liệu so sánh đối chuẩn (Dummy vs Random Forest vs Logistic Regression), phân tích tầm quan trọng của các đặc trưng học tập (VLE interactions, assessments, prior retakes, credits), và đánh giá độ tin cậy xác suất phục vụ cố vấn học vụ.


In [2]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

# Đường dẫn artifact được phê duyệt
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'artifacts').exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent

ARTIFACT_DIR = REPO_ROOT / 'artifacts' / 'approved' / 'final-002'
if not ARTIFACT_DIR.exists():
    ARTIFACT_DIR = Path('/artifacts/approved/final-002')

with open(ARTIFACT_DIR / 'manifest.json', 'r', encoding='utf-8') as f:
    manifest = json.load(f)

with open(ARTIFACT_DIR / 'metrics.json', 'r', encoding='utf-8') as f:
    metrics = json.load(f)

with open(ARTIFACT_DIR / 'explanation_config.json', 'r', encoding='utf-8') as f:
    explanation_config = json.load(f)

print(f"=== MÔ HÌNH: {manifest['model_id']} ({manifest['model_version']}) ===")
print(f"Domain: {manifest['domain_id']} | Cutoff: Ngày {manifest['cutoff_day']} | Target: {manifest['target_id']}")
print(f"Huấn luyện lúc: {manifest['trained_at']} | Source commit: {manifest['source_commit']}")


=== MÔ HÌNH: oulad-risk-logistic-regression (final-002) ===
Domain: oulad | Cutoff: Ngày 28 | Target: non_completion_at_end
Huấn luyện lúc: 2026-09-06T08:36:19.099827+00:00 | Source commit: e388e7744e85c7a44a51c0954b7d2c92decf68fb


## 1. Bảng so sánh các mô hình trên tập phát triển (Dev Set Validation)

Quy mô tập Dev: **349 sinh viên** (Tỷ lệ rớt/rút môn thực tế: **40.40%**).  
Quy tắc chọn mô hình: Lựa chọn mô hình đạt **Average Precision (AP)** cao nhất trên Dev; tối ưu ngưỡng cảnh báo (*Alert Threshold*) theo F1 score.


In [4]:
comparison_rows = []
for item in metrics['comparison_dev']:
    m = item['metrics']
    comparison_rows.append({
        'Mô hình': item['model'].replace('_', ' ').title(),
        'Average Precision (AP)': round(m['average_precision'], 4),
        'ROC-AUC': round(m['roc_auc'], 4),
        'F1-Score': round(m['f1'], 4),
        'Recall (Độ nhạy)': round(m['recall'], 4),
        'Precision (Độ chuẩn)': round(m['precision'], 4),
        'Brier Score (MSE)': round(m['brier'], 4),
        'Ngưỡng tối ưu (Threshold)': round(m['threshold'], 4)
    })

df_comparison = pd.DataFrame(comparison_rows)
df_comparison


,Mô hình,Average Precision (AP),ROC-AUC,F1-Score,Recall (Độ nhạy),Precision (Độ chuẩn),Brier Score (MSE),Ngưỡng tối ưu (Threshold)
0,Dummy,0.4040,0.5000,0.5755,1.0000,0.4040,0.2418,0.43
1,Logistic Regression,0.6348,0.7152,0.6442,0.8794,0.5082,0.2082,0.32
2,Random Forest,0.6180,0.6672,0.6031,0.8298,0.4737,0.2194,0.33


### Nhận xét đánh giá Dev Set:
- **Logistic Regression** vượt trội cả **Dummy Baseline** (+57.1% AP) và **Random Forest** (+2.7% AP).
- **Brier Score** của Logistic Regression đạt 0.2082 (thấp nhất trong cả 3 mô hình), phản ánh sai số bình phương xác suất nhỏ nhất.
- **Recall đạt 87.94%** ở ngưỡng cảnh báo `threshold = 0.32`, đảm bảo phần lớn sinh viên có nguy cơ rớt môn đều được phát hiện sớm.


## 2. Đánh giá tổng quát hóa trên tập kiểm thử độc lập (Test Set Evaluation)

Tập Test gồm **329 sinh viên** hoàn toàn tách biệt (*student-disjoint split*, seed 42) để kiểm chứng khả năng tổng quát hóa, không bị rò rỉ dữ liệu (*data leakage*).


In [5]:
t = metrics['test_metrics']
test_summary = pd.DataFrame([{
    'Số mẫu kiểm thử (N)': t['sample_count'],
    'Tỷ lệ rủi ro thực tế': f"{round(t['positive_prevalence'] * 100, 2)}%",
    'Average Precision (AP)': round(t['average_precision'], 4),
    'ROC-AUC': round(t['roc_auc'], 4),
    'Recall': round(t['recall'], 4),
    'Precision': round(t['precision'], 4),
    'F1-Score': round(t['f1'], 4),
    'Brier Score': round(t['brier'], 4)
}])
test_summary


,Số mẫu kiểm thử (N),Tỷ lệ rủi ro thực tế,Average Precision (AP),ROC-AUC,Recall,Precision,F1-Score,Brier Score
0,329,43.16%,0.5892,0.6385,0.8028,0.475,0.5969,0.231


In [6]:
cm = np.array(t['confusion_matrix'])
df_cm = pd.DataFrame(cm, 
    index=['Thực tế: ĐẬU/HOÀN THÀNH (0)', 'Thực tế: RỚT/RÚT MÔN (1)'],
    columns=['Dự đoán: ĐẬU (0)', 'Dự đoán: RỚT (1)'])
print('=== MA TRẬN NHẦM LẪN (CONFUSION MATRIX TRÊN TẬP TEST N=329) ===')
print(f"- True Negatives (TN): {cm[0, 0]} (Dự đoán Đậu - Thực tế Đậu)")
print(f"- False Positives (FP): {cm[0, 1]} (Cảnh báo nhầm - Thực tế Đậu)")
print(f"- False Negatives (FN): {cm[1, 0]} (Bỏ sót - Thực tế Rớt)")
print(f"- True Positives (TP): {cm[1, 1]} (Cảnh báo đúng - Thực tế Rớt)")
df_cm


=== MA TRẬN NHẦM LẪN (CONFUSION MATRIX TRÊN TẬP TEST N=329) ===
- True Negatives (TN): 61 (Dự đoán Đậu - Thực tế Đậu)
- False Positives (FP): 126 (Cảnh báo nhầm - Thực tế Đậu)
- False Negatives (FN): 28 (Bỏ sót - Thực tế Rớt)
- True Positives (TP): 114 (Cảnh báo đúng - Thực tế Rớt)


,Dự đoán: ĐẬU (0),Dự đoán: RỚT (1)
Thực tế: ĐẬU/HOÀN THÀNH (0),61,126
Thực tế: RỚT/RÚT MÔN (1),28,114


## 3. Phân tích đặc trưng & Trọng số mô hình (Feature Analysis & Weights)

Các đặc trưng được trích xuất từ dữ liệu OULAD tại mốc Ngày 28 và chuẩn hóa qua `StandardScaler`.
Hệ số hồi quy (*Regression Coefficient*) biểu thị mức độ ảnh hưởng đến Log-odds của nguy cơ rớt môn.


In [7]:
import joblib
pipeline = joblib.load(ARTIFACT_DIR / 'pipeline.joblib')
scaler = pipeline.named_steps['scaler']
clf = pipeline.named_steps['classifier']

features = manifest['feature_order']
weights = clf.coef_[0]
means = scaler.mean_
stds = scaler.scale_

descriptions = [
    'Số lần đã từng học lại môn này trước đây',
    'Tổng số tín chỉ đăng ký học đồng thời trong kỳ',
    'Tổng số lượt click trên hệ thống VLE (ngày 0-28)',
    'Số ngày có tương tác VLE (ngày 0-28)',
    'Số bài đánh giá/bài tập đã nộp (ngày 0-28)',
    'Số ngày trôi qua từ lần tương tác VLE cuối đến ngày 28'
]

df_features = pd.DataFrame({
    'Đặc trưng (Feature)': features,
    'Mô tả': descriptions,
    'Hệ số hồi quy (Coefficient)': [round(w, 4) for w in weights],
    'Tác động': ['Tăng rủi ro (+)' if w > 0 else 'Giảm rủi ro (-)' for w in weights],
    'Giá trị trung bình (Mean)': [round(m, 2) for m in means],
    'Độ lệch chuẩn (Std)': [round(s, 2) for s in stds]
}).sort_values(by='Hệ số hồi quy (Coefficient)', key=abs, ascending=False)

print(f"Điểm chặn mô hình (Intercept): {round(clf.intercept_[0], 4)}")
df_features


Điểm chặn mô hình (Intercept): -0.2928


,Đặc trưng (Feature),Mô tả,Hệ số hồi quy (Coefficient),Tác động,Giá trị trung bình (Mean),Độ lệch chuẩn (Std)
3,vle_active_days_0_28,Số ngày có tương tác VLE (ngày 0-28),-0.5730,Giảm rủi ro (-),10.65,7.29
4,assessment_submitted_count_0_28,Số bài đánh giá/bài tập đã nộp (ngày 0-28),-0.2185,Giảm rủi ro (-),0.86,0.64
0,num_of_prev_attempts,Số lần đã từng học lại môn này trước đây,0.2178,Tăng rủi ro (+),0.16,0.51
1,studied_credits,Tổng số tín chỉ đăng ký học đồng thời trong kỳ,0.1984,Tăng rủi ro (+),75.14,36.37
2,vle_clicks_0_28,Tổng số lượt click trên hệ thống VLE (ngày 0-28),0.0593,Tăng rủi ro (+),276.78,343.92
5,days_since_last_vle_activity,Số ngày trôi qua từ lần tương tác VLE cuối đến...,0.0445,Tăng rủi ro (+),3.59,4.92


### Phân tích ý nghĩa học vụ:
1. **`vle_active_days_0_28` (Hệ số: -0.5730)**: Đặc trưng bảo vệ có tác động mạnh nhất. Sinh viên vào học đều đặn nhiều ngày (thói quen học tập phân bổ) có nguy cơ rớt môn giảm rất mạnh.
2. **`assessment_submitted_count_0_28` (Hệ số: -0.2185)**: Nộp bài kiểm tra/bài tập sớm trong 4 tuần đầu là chỉ dấu tích cực cho cam kết hoàn thành khóa học.
3. **`num_of_prev_attempts` (Hệ số: +0.2178)**: Đã từng học lại là tín hiệu cảnh báo rủi ro cao. Nhóm này cần được cố vấn theo dõi sát sao.
4. **`studied_credits` (Hệ số: +0.1984)**: Tải học tập quá nặng (nhiều tín chỉ cùng lúc) làm tăng xác suất bỏ môn do quá tải.


## 4. Đánh giá độ hiệu chuẩn xác suất (Probability Calibration)

Độ hiệu chuẩn thể hiện sự tương quan giữa xác suất mô hình dự đoán và tỷ lệ sinh viên thực tế gặp rủi ro trong từng phân nhóm.


In [1]:
cal = metrics['test_metrics']['calibration']
df_cal = pd.DataFrame({
    'Xác suất dự đoán trung bình': [round(p, 4) for p in cal['mean_predicted_probability']],
    'Tỷ lệ rủi ro thực tế': [round(f, 4) for f in cal['fraction_positive']]
})
print('=== BẢNG HIỆU CHUẨN XÁC SUẤT TRÊN TẬP TEST ===')
df_cal


NameError: name 'metrics' is not defined

## 5. Ranh giới sử dụng an toàn (Contract 1.0.0 & Governance Policy)

- **Chỉ dùng cho mục đích nghiên cứu OULAD**: Mô hình `final-002` được phê duyệt phục vụ đối chuẩn nghiên cứu công khai.
- **Không áp dụng suy luận cho sinh viên thực tế (NTTU / DEMO-1)**: Khi hệ thống nhận diện yêu cầu dự đoán cho sinh viên DEMO, hệ thống sẽ từ chối an toàn với mã `409 MODEL_DOMAIN_MISMATCH` nhằm ngăn chặn thiên lệch mô hình giữa môi trường học từ xa Open University tại Anh và chương trình đào tạo tín chỉ tại Việt Nam.
- **Minh bạch giải thích**: Các đóng góp Log-odds được biểu diễn rõ ràng qua Linear Contributions/SHAP, không suy diễn quan hệ nhân quả tuyệt đối.
